# 01 SFT
Attach: repo zip dataset. Accelerator: GPU T4 x2. Internet: on.

In [ ]:
import glob, os, shutil, zipfile
matches = glob.glob("/kaggle/input/**/submission_adapter.py", recursive=True)
if not matches:
    for archive in glob.glob("/kaggle/input/**/*.zip", recursive=True):
        zipfile.ZipFile(archive).extractall("/tmp/unzipped")
    matches = glob.glob("/tmp/unzipped/**/submission_adapter.py", recursive=True)
assert matches, "attach the repo zip as a Kaggle dataset"
REPO = "/tmp/repo"
shutil.rmtree(REPO, ignore_errors=True)
shutil.copytree(os.path.dirname(matches[0]), REPO)
os.chdir(REPO)
os.environ["PYTHONPATH"] = REPO
OUT = "/kaggle/working/outputs"
os.makedirs(OUT, exist_ok=True)
print(os.listdir(REPO))

In [ ]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest
!python -m pip install -q -e . --no-deps

In [ ]:
!nvidia-smi
import torch, transformers, datasets
print("torch", torch.__version__, "transformers", transformers.__version__, "datasets", datasets.__version__)
print("gpus", torch.cuda.device_count())

## Data

In [ ]:
DATA = f"{OUT}/data/sft"
!python scripts/prepare_sft_data.py --output_dir {DATA}

## Smoke test (5 steps)

In [ ]:
!python -m alignment.train_sft --train_path {DATA}/sft_train.jsonl --val_path {DATA}/sft_val.jsonl --output_dir /tmp/smoke_sft --max_train_examples 160 --max_val_examples 32 --max_steps 5 --eval_every 5 --log_every 1

## Full run

In [ ]:
!python -m alignment.train_sft --train_path {DATA}/sft_train.jsonl --val_path {DATA}/sft_val.jsonl --output_dir {OUT}/sft

In [ ]:
!cd /kaggle/working && zip -rq sft_run_outputs.zip outputs/sft outputs/data
!ls -lh /kaggle/working